# MLflow Models: Model Flavors and Pipelines

In this notebook we will cover:
1. What are Model Flavors
2. Built-in flavors (sklearn, xgboost, lightgbm)
3. PyFunc flavor (universal)
4. Scikit-learn pipelines
5. Custom transformers

## What are Model Flavors?

**Model Flavor** is a way MLflow stores and loads models for different frameworks:
- `sklearn` - scikit-learn models
- `tensorflow` - TensorFlow/Keras models
- `pytorch` - PyTorch models
- `xgboost` - XGBoost models
- `lightgbm` - LightGBM models
- `pyfunc` - universal Python function (any model)

## 1. Import Libraries

In [1]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import mlflow.lightgbm
import mlflow.pyfunc
from mlflow.models import infer_signature

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import xgboost as xgb
import lightgbm as lgb
import sys

sys.path.append('../src')
from data_loader import load_sample_data
from model_inspector import inspect_model_structure, compare_models

print(f"MLflow version: {mlflow.__version__}")
print(f"XGBoost version: {xgb.__version__}")
print(f"LightGBM version: {lgb.__version__}")

MLflow version: 3.9.0
XGBoost version: 3.2.0
LightGBM version: 4.6.0


## 2. Data Preparation

In [2]:
X_train, X_test, y_train, y_test, feature_names, target_names = load_sample_data('breast_cancer')

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Number of features: {len(feature_names)}")
print(f"Classes: {target_names}")

Training set: (455, 30)
Test set: (114, 30)
Number of features: 30
Classes: ['malignant' 'benign']


## 3. Scikit-learn Flavor

In [3]:
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("03_Model_Flavors")

with mlflow.start_run(run_name="sklearn_flavor") as run:
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    X_sample = pd.DataFrame(X_train[:5], columns=feature_names)
    y_pred = model.predict(X_sample)
    signature = infer_signature(X_sample, y_pred)
    
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model",
        signature=signature
    )
    
    run_id_sklearn = run.info.run_id
    accuracy = model.score(X_test, y_test)
    mlflow.log_metric("accuracy", accuracy)
    
    print(f"Run ID: {run_id_sklearn}")
    print(f"Accuracy: {accuracy:.4f}")

c:\Learn\github\mlpops-basics\.venv\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:178: FutureWarning: The filesystem tracking backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance. For migrating existing data, https://github.com/mlflow/mlflow-export-import can be used.
  return FileStore(store_uri, store_uri)
2026/02/15 01:51:01 INFO mlflow.tracking.fluent: Experiment with name '03_Model_Flavors' does not exist. Creating a new experiment.
c:\Learn\github\mlpops-basics\.venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
2026/02/15 01:51:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
c:\Learn\git

Run ID: 705dafc2dc64418c80cd940dae5f4959
Accuracy: 0.9561


## 4. XGBoost Flavor

In [5]:
feature_names_list = list(feature_names)

with mlflow.start_run(run_name="xgboost_flavor") as run:
    dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=feature_names_list)
    dtest = xgb.DMatrix(X_test, label=y_test, feature_names=feature_names_list)
    
    params = {
        'max_depth': 5,
        'eta': 0.1,
        'objective': 'binary:logistic',
        'eval_metric': 'logloss'
    }
    
    model = xgb.train(params, dtrain, num_boost_round=100)
    
    X_sample = pd.DataFrame(X_train[:5], columns=feature_names)
    y_pred = model.predict(xgb.DMatrix(X_sample, feature_names=feature_names_list))
    signature = infer_signature(X_sample, y_pred)
    
    mlflow.xgboost.log_model(
        xgb_model=model,
        artifact_path="model",
        signature=signature
    )
    
    run_id_xgb = run.info.run_id
    y_pred_test = model.predict(dtest)
    accuracy = ((y_pred_test > 0.5).astype(int) == y_test).mean()
    mlflow.log_metric("accuracy", accuracy)
    
    print(f"Run ID: {run_id_xgb}")
    print(f"Accuracy: {accuracy:.4f}")

2026/02/15 01:51:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Run ID: db340161112d43fb8bfb8ca5220bcc92
Accuracy: 0.9474


## 5. LightGBM Flavor

In [6]:
with mlflow.start_run(run_name="lightgbm_flavor") as run:
    train_data = lgb.Dataset(X_train, label=y_train, feature_name=list(feature_names))
    
    params = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'num_leaves': 31,
        'learning_rate': 0.05,
        'feature_fraction': 0.9
    }
    
    model = lgb.train(params, train_data, num_boost_round=100)
    
    X_sample = pd.DataFrame(X_train[:5], columns=feature_names)
    y_pred = model.predict(X_sample)
    signature = infer_signature(X_sample, y_pred)
    
    mlflow.lightgbm.log_model(
        lgb_model=model,
        artifact_path="model",
        signature=signature
    )
    
    run_id_lgb = run.info.run_id
    y_pred_test = model.predict(X_test)
    accuracy = ((y_pred_test > 0.5).astype(int) == y_test).mean()
    mlflow.log_metric("accuracy", accuracy)
    
    print(f"Run ID: {run_id_lgb}")
    print(f"Accuracy: {accuracy:.4f}")

2026/02/15 01:51:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 285, number of negative: 170
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000245 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4542
[LightGBM] [Info] Number of data points in the train set: 455, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.626374 -> initscore=0.516691
[LightGBM] [Info] Start training from score 0.516691
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 

## 6. Scikit-learn Pipeline

In [7]:
with mlflow.start_run(run_name="sklearn_pipeline") as run:
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('pca', PCA(n_components=10)),
        ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
    ])
    
    pipeline.fit(X_train, y_train)
    
    X_sample = pd.DataFrame(X_train[:5], columns=feature_names)
    y_pred = pipeline.predict(X_sample)
    signature = infer_signature(X_sample, y_pred)
    
    mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="model",
        signature=signature
    )
    
    run_id_pipeline = run.info.run_id
    accuracy = pipeline.score(X_test, y_test)
    mlflow.log_metric("accuracy", accuracy)
    
    print(f"Run ID: {run_id_pipeline}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"\nPipeline steps: {pipeline.named_steps.keys()}")

c:\Learn\github\mlpops-basics\.venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
2026/02/15 01:52:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
c:\Learn\github\mlpops-basics\.venv\Lib\site-packages\mlflow\models\model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)


Run ID: 59fc982546d94a03a2b3e41e69ff0a96
Accuracy: 0.9211

Pipeline steps: dict_keys(['scaler', 'pca', 'classifier'])


## 7. PyFunc Flavor - Universal Wrapper

In [8]:
class CustomModel(mlflow.pyfunc.PythonModel):
    """
    Simple ensemble: average predictions from multiple models
    """
    
    def __init__(self, models):
        self.models = models
    
    def predict(self, context, model_input):
        predictions = []
        for model in self.models:
            pred = model.predict(model_input)
            predictions.append(pred)
        
        avg_predictions = np.mean(predictions, axis=0)
        return (avg_predictions > 0.5).astype(int)

rf_model = RandomForestClassifier(n_estimators=50, random_state=42)
rf_model.fit(X_train, y_train)

from sklearn.linear_model import LogisticRegression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

custom_model = CustomModel(models=[rf_model, lr_model])

c:\Learn\github\mlpops-basics\.venv\Lib\site-packages\mlflow\pyfunc\utils\data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
c:\Learn\github\mlpops-basics\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [9]:
with mlflow.start_run(run_name="pyfunc_ensemble") as run:
    X_sample = pd.DataFrame(X_train[:5], columns=feature_names)
    y_pred = custom_model.predict(None, X_sample)
    signature = infer_signature(X_sample, y_pred)
    
    mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=custom_model,
        signature=signature
    )
    
    run_id_pyfunc = run.info.run_id
    
    X_test_df = pd.DataFrame(X_test, columns=feature_names)
    y_pred_test = custom_model.predict(None, X_test_df)
    accuracy = (y_pred_test == y_test).mean()
    mlflow.log_metric("accuracy", accuracy)
    
    print(f"Run ID: {run_id_pyfunc}")
    print(f"Ensemble Accuracy: {accuracy:.4f}")

c:\Learn\github\mlpops-basics\.venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
c:\Learn\github\mlpops-basics\.venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
2026/02/15 01:52:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
c:\Learn\github\mlpops-basics\.venv\Lib\site-packages\mlflow\tracing\provider.py:757: FutureWarning: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
  result = f(*args, **kwargs)
c:\Learn\github\mlpops-basics\.v

Run ID: 397c3696730b48bfb140b43b2ded033a
Ensemble Accuracy: 0.9474


c:\Learn\github\mlpops-basics\.venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
c:\Learn\github\mlpops-basics\.venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


## 8. Loading Models with Different Flavors

### Load sklearn model

In [10]:
loaded_sklearn = mlflow.sklearn.load_model(f"runs:/{run_id_sklearn}/model")
print(f"Model type: {type(loaded_sklearn)}")
print(f"Prediction: {loaded_sklearn.predict(X_test[:1])}")

Model type: <class 'sklearn.ensemble._forest.RandomForestClassifier'>
Prediction: [0]


### Load XGBoost model

In [12]:
loaded_xgb = mlflow.xgboost.load_model(f"runs:/{run_id_xgb}/model")
print(f"Model type: {type(loaded_xgb)}")
X_test_dmatrix = xgb.DMatrix(X_test[:1], feature_names=feature_names_list)
print(f"Prediction: {loaded_xgb.predict(X_test_dmatrix)}")

Model type: <class 'xgboost.core.Booster'>
Prediction: [0.00128752]


### Load ANY model as PyFunc

In [13]:
X_test_df = pd.DataFrame(X_test[:1], columns=feature_names)

print("Loading sklearn model as pyfunc:")
loaded_pyfunc_sklearn = mlflow.pyfunc.load_model(f"runs:/{run_id_sklearn}/model")
print(f"Prediction: {loaded_pyfunc_sklearn.predict(X_test_df)}")

print("\nLoading XGBoost model as pyfunc:")
loaded_pyfunc_xgb = mlflow.pyfunc.load_model(f"runs:/{run_id_xgb}/model")
print(f"Prediction: {loaded_pyfunc_xgb.predict(X_test_df)}")

print("\nLoading LightGBM model as pyfunc:")
loaded_pyfunc_lgb = mlflow.pyfunc.load_model(f"runs:/{run_id_lgb}/model")
print(f"Prediction: {loaded_pyfunc_lgb.predict(X_test_df)}")

Loading sklearn model as pyfunc:
Prediction: [0]

Loading XGBoost model as pyfunc:
Prediction: [0.00128752]

Loading LightGBM model as pyfunc:


c:\Learn\github\mlpops-basics\.venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


Prediction: [0.00433945]


## 9. Comparing Different Flavors

In [14]:
print("Sklearn model structure:")
inspect_model_structure(f"runs:/{run_id_sklearn}/model", verbose=False)

print("\n" + "="*60)
print("XGBoost model structure:")
inspect_model_structure(f"runs:/{run_id_xgb}/model", verbose=False)

print("\n" + "="*60)
print("PyFunc model structure:")
inspect_model_structure(f"runs:/{run_id_pyfunc}/model", verbose=False)

Sklearn model structure:

XGBoost model structure:

PyFunc model structure:


{'local_path': 'C:\\Users\\PPOBER~1\\AppData\\Local\\Temp\\tmpq8_807dg\\model\\',
 'files': ['conda.yaml',
  'MLmodel',
  'python_env.yaml',
  'python_model.pkl',
  'requirements.txt'],
 'mlmodel': {'artifact_path': 'file:///c:/Learn/github/mlpops-basics/03_mlflow_models/notebooks/mlruns/569162965248699545/models/m-1e8114a91c68461fb0da98420fbf9abe/artifacts',
  'flavors': {'python_function': {'cloudpickle_version': '3.1.2',
    'code': None,
    'env': {'conda': 'conda.yaml', 'virtualenv': 'python_env.yaml'},
    'loader_module': 'mlflow.pyfunc.model',
    'python_model': 'python_model.pkl',
    'python_version': '3.11.6',
    'streamable': False}},
  'is_signature_from_type_hint': False,
  'mlflow_version': '3.9.0',
  'model_id': 'm-1e8114a91c68461fb0da98420fbf9abe',
  'model_size_bytes': 159960,
  'model_uuid': 'm-1e8114a91c68461fb0da98420fbf9abe',
  'prompts': None,
  'run_id': '397c3696730b48bfb140b43b2ded033a',
  'signature': {'inputs': '[{"type": "double", "name": "mean radius", 

## Conclusions

In this notebook we learned:

1.  Different MLflow model flavors (sklearn, xgboost, lightgbm)
2.  Logging models from different frameworks
3.  Working with scikit-learn pipelines
4.  Creating custom PyFunc models
5.  Universal PyFunc interface for all models
6.  Comparing model structures across flavors

### Key Takeaways:
- Each flavor has its own `log_model()` and `load_model()` functions
- PyFunc is universal - works with ANY model
- Pipelines are logged as sklearn models
- Custom models need PyFunc wrapper

### Next Steps:
- **Notebook 04**: Advanced Custom PyFunc Models
- **Notebook 05**: Model Deployment